# Допуск: CollabLLM: From Passive Responders to Active Collaborators

**Статья:** S. Wu, M. Galley, B. Peng, H. Cheng, G. Li, Y. Dou, W. Cai, J. Zou, J. Leskovec,
J. Gao, 2025 — <https://arxiv.org/abs/2502.00640>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций заменены: определите, что чему соответствует, и
  ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **перепутаны местами блоки кода** — какой-то блок стоит не на своём месте: найдите его;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его;
- **вопрос по коду** — в коде нет отклонений: ответьте на вопросы, опираясь на код и статью.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / приложение / цитата) нарушено и как должно быть. Код писать не нужно —
достаточно объяснить словами. Запускать и отлаживать код тоже не нужно: все задания решаются
чтением кода и статьи.

Общие импорты:

In [ ]:
import json
from statistics import mean

Диалог `conv` — список сообщений `{"role": "user" | "assistant", "content": str}`. У цели
пользователя `goal` есть `goal.query` (полный запрос, который видит только симулятор
пользователя) и `goal.reference` (эталон: документ, тесты или ответ). Сниппеты — части одной
реализации; функции, которые используются, но не показаны, считайте корректными:

- `policy(conv)` — ответ обучаемой модели на последнее сообщение;
- `user_sim(conv, goal)` — реплика симулятора пользователя или `TERMINATE`;
- `extract_solution(conv)` — итоговое решение из диалога;
- `task_metric(solution, reference)` — BLEU / доля пройденных тестов / точность;
- `intrinsic(conv)` — внутренняя награда R_int из статьи.

## Задание 1. Четыре способа оценить ответ

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

`history` — диалог до текущего хода (заканчивается репликой пользователя), `response` —
кандидат на ответ модели. Определите, какая функция — награда **Helpfulness**, **Extrinsic**,
**Extrinsic + Intrinsic** и **Multiturn-aware Reward** (§3.1, §5.1). Ошибок в коде нет.

In [ ]:
def score_a(history, response, goal):
    return judge_helpfulness(goal.query, history, response)          # LLM judge, 0..1

def score_b(history, response, goal):
    conv = history + [response]
    return task_metric(extract_solution(conv), goal.reference)

def score_c(history, response, goal):
    conv = history + [response]
    return task_metric(extract_solution(conv), goal.reference) + intrinsic(conv)

def score_d(history, response, goal, w=2, n=3):
    total = 0.0
    for _ in range(n):
        conv = history + [response]
        for _ in range(w):
            u = user_sim(conv, goal)
            if u == TERMINATE:
                break
            conv = conv + [u, policy(conv + [u])]
        total += task_metric(extract_solution(conv), goal.reference) + intrinsic(conv)
    return total / n

**Вопросы.**
1. Рис. 5–6: пользователь просит функцию токенизации файла. Ответ A сразу пишет код и сам
   решает убрать пунктуацию и стоп-слова; ответ B задаёт три уточняющих вопроса. Какие
   функции предпочтут A, какие B и почему? Почему одна из функций даёт обоим ответам 0?
2. Одна из функций — частный случай другой. Какой и при каком значении параметра? Почему,
   по рис. 4, она уступает «старшей»?
3. Что определяет стоимость `score_d`? Какие значения параметров выбрали авторы и чем
   объясняют выбор (§5.1, прил. B.2, табл. 6–7)?
4. Авторы называют свой подход каузальным, а награду траектории в MTPO и похожих работах —
   наблюдательной (прил. A). Какая строка `score_d` делает оценку «интервенционной»: что
   здесь фиксируется, а что сэмплируется?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Награда за весь диалог

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

Награда R*(t | g) для готового диалога (формулы 2–4). `tok` — токенизатор. Найдите одно
отклонение от §3.1.1.

In [ ]:
def conversation_reward(conv, goal, tok, lam=5e-4):
    r_ext = task_metric(extract_solution(conv), goal.reference)        # extrinsic
    n_tokens = sum(len(tok(m["content"]).input_ids) for m in conv)
    r_eff = -min(lam * n_tokens, 1.0)                                   # efficiency
    n_questions = sum(m["content"].count("?") for m in conv if m["role"] == "assistant")
    r_itr = min(n_questions / 5, 1.0)                                   # interactivity, 0..1
    return r_ext + r_eff + r_itr

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Синтетические диалоги для SFT и DPO

**Тип задания:** перепутаны местами блоки кода.

Построение обучающих данных по прил. B.1 и рис. 8. `multiturn_reward(conv, response, goal)`
возвращает пару (MR ответа, сэмплированное продолжение диалога после него — список
сообщений, начинающийся с реплики пользователя). Найдите одно отклонение.

In [ ]:
def collect_conversation(goal, n_cand=3, max_turns=8):
    conv = [{"role": "user", "content": user_sim([], goal)}]
    sft, dpo = [], []
    for _ in range(max_turns):
        cands = [policy(conv) for _ in range(n_cand)]                   # sampled, T = 1
        scored = [(*multiturn_reward(conv, c, goal), c) for c in cands]
        scored.sort(key=lambda x: x[0])
        (_, _, worst), (_, future, best) = scored[0], scored[-1]
        conv = conv + [best, future[0]]          # continue with the best response and the
                                                 # user's next turn from its own rollout
        sft.append({"prompt": conv, "response": best})
        dpo.append({"prompt": conv, "chosen": best, "rejected": worst})
        if future[0]["content"] == TERMINATE:
            break
    return sft, dpo

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Неоднозначные вопросы Abg-CoQA

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Проверка обобщения из §5.3: модель, обученная на коде, отвечает на вопросы по рассказу; часть
вопросов неоднозначна. `asks_question(reply)` определяет, задал ли ответ уточняющий вопрос.
Найдите один неверный комментарий.

In [ ]:
TEMPLATE = "Can you help me answer a question about the following story?\n\n{story}\n\nMy question is: {q}"

def action_accuracy(model, data):
    # data: list of (story, question, is_ambiguous)
    hits = {True: [], False: []}
    for story, q, ambiguous in data:
        reply = model(TEMPLATE.format(story=story, q=q))
        hits[ambiguous].append(asks_question(reply) == ambiguous)   # ask iff ambiguous
    acc_amb, acc_non = mean(hits[True]), mean(hits[False])
    # macro accuracy averages the two classes; a model that never asks still scores about
    # 0.95 on it (like GPT-4o), so macro accuracy cannot separate passive and active models
    return acc_amb, acc_non, (acc_amb + acc_non) / 2

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Симулятор пользователя

**Тип задания:** вопрос по коду — в коде нет отклонений, ответьте на вопросы.

Сокращённая версия промпта из прил. D.1. `llm(prompt)` — вызов модели-симулятора (в статье
GPT-4o-mini), `format_history(conv)` — диалог в виде текста.

In [ ]:
GUIDELINES = [
    "Minimize Effort: avoid being too detailed. Provide vague or incomplete demands in the early "
    "stages; let the AI ask for clarification rather than providing everything upfront.",
    "Occasionally Make Mistakes: misspell words, give wrong information, ask unclear questions.",
    "Mention Personal Preferences, e.g. 'I prefer short answers'.",
    "Goal-Oriented: keep the chat focused on your intent, avoid small talk.",
]

def simulate_user(llm, task_desc, goal, conv, terminal="[[TERMINATE CHAT]]"):
    prompt = ("You are role-playing as a human USER interacting with an AI collaborator.\n"
              f"Task: {task_desc}\n"
              f"Complete prompt or reference goal (not visible to the AI): {goal.query}\n"
              f"Chat history:\n{format_history(conv)}\n"
              "Guidelines:\n" + "\n".join(f"- {g}" for g in GUIDELINES) + "\n"
              'Output JSON with "current_answer", "thought" and "response". '
              f'Use "{terminal}" as the response when your goal is solved.')
    return json.loads(llm(prompt))["response"]

**Вопросы.**
1. Что произойдёт с Multiturn-aware Reward и с обучением, если убрать первое правило и
   симулятор будет сразу выкладывать полный запрос `goal.query`? Какое поведение модели
   тогда окажется выгоднее — уточнять или сразу отвечать?
2. Зачем симулятору знать `goal.query`, если ассистент его не видит? Где в формуле MR
   (формула 1) используется цель g?
3. Почему авторы не используют открытую модель в роли симулятора (прил. B.2)? Чем
   симулированные пользователи отличаются от настоящих (прил. F.2) и как авторы проверяли,
   что модель, обученная на симуляторе, полезна живым людям (§6)?

**Ваш ответ:**

_Напишите ответ здесь._